## DATA2-173

In [0]:
env="p"

In [0]:

spark.sql(f"""ALTER TABLE `{env}-cdh`.primaryactive.Interaction
ADD COLUMN IsRPCFlagCalc BOOLEAN AFTER IsRPCFlag;
""")

## RBTC-121

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.payment
ADD COLUMN PaymentTypeGroup STRING AFTER PaymentRecordType;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.Payment
ADD COLUMN PaymentTypeEng STRING AFTER PaymentTypeGroup;"""
)

In [0]:
spark.sql(f"""        
MERGE INTO `{env}-cdh`.primaryactive.payment p
USING `{env}-cdh`.reference.paymenttypegroupmapping gm
ON gm.PaymentTypeCode = p.ProfitsPaymentTypeCode
AND p.sourcesystem="profits"
WHEN MATCHED THEN
UPDATE SET
p.PaymentTypeEng = cast(gm.ProfitsPaymentType as string) 
""")

##DBTC-944

In [0]:
spark.sql(f"""alter table `{env}-cdh`.primaryactive.trackingstep
add column SolutionREStatus string AFTER Comments;
""")

In [0]:
spark.sql(f"""
MERGE INTO `{env}-cdh`.primaryactive.trackingstep t
USING `{env}-cdh`.landed.sme_trackingstep st
ON t.TrackingStepId=st.DebtorStatusReColActionID 
and t.BusinessDate=st.BusinessDate
WHEN MATCHED THEN
UPDATE SET
t.SolutionREStatus = cast(st.Step_status as string)
""")

### Delete businessdate "2024-04-05" from landed,primaryactive,derivativeactive according to ticket DBTC-915

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize SparkSession
spark = SparkSession.builder.appName("Delete20240405").getOrCreate()

# List of directories (catalogs and schemas) you want to process
directories = [
    {"catalog": "`p-cdh`", "schema": "primaryactive"},
    {"catalog": "`p-cdh`", "schema": "derivativeactive"},
    {"catalog": "`p-cdh`", "schema": "landed"}
]

# Initialize a list to store results
results = []

# Iterate over each directory (catalog, schema combination)
for directory in directories:
    catalog_name = directory["catalog"]
    schema_name = directory["schema"]

    # Get the list of tables in the specified catalog and schema
    tables_df = spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}")

    # Collect table names as a list using DataFrame operations
    tables = tables_df.filter(col("isTemporary") == False).select("tableName").collect()
    table_names = [row["tableName"] for row in tables]

    # Iterate over each table in the current directory
    for table_name in table_names:
        try:
            # Fully qualified table name in Unity Catalog
            full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

            # Load the table
            table_df = spark.table(full_table_name)

            # Check if the required columns exist
            if "BusinessDate" in table_df.columns:
                # Perform the DELETE operation
                delete_query = f"DELETE FROM {full_table_name} WHERE BusinessDate = '2024-04-05'"
                spark.sql(delete_query)

                # Log the result
                results.append((table_name, catalog_name, schema_name, "Deleted records with BusinessDate = '2024-04-05'"))
                print(f"Success: Deleted records from table {table_name} in {catalog_name}.{schema_name} where BusinessDate = '2024-04-05'")
            else:
                print(f"Skipping table {table_name} in {catalog_name}.{schema_name} as it does not contain required columns.")
        except Exception as e:
            print(f"Error processing table {table_name} in {catalog_name}.{schema_name}: {e}")



## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

